# Qwen2.5 Premise Generator Training Notebook

Fine-tunes **Qwen2.5-3B-Instruct** using **LoRA / QLoRA** for the **Property Law Premise Generator** component.

**Task learned by this model:**

`property-law topic → realistic Indian property-law factual premise`

This is separate from the Opposing Counsel model. This model generates realistic case backgrounds; the Opposing Counsel model later challenges arguments based on those facts.

Pipeline features:
- JSONL dataset validation
- System prompt normalization
- ChatML/messages training format
- Qwen chat template usage
- 4-bit QLoRA loading
- Assistant-only loss
- Train/eval split
- Resume-safe training
- Adapter saving
- Post-training inference test

In [ ]:
# =========================
# INSTALL DEPENDENCIES
# =========================
# Run this cell once. On Colab, you may skip the PyTorch reinstall if CUDA is already working.

# Install CUDA-enabled PyTorch first (Windows, CUDA 12.1 wheels)
!pip uninstall -y torch torchvision torchaudio
!pip install -q --index-url https://download.pytorch.org/whl/cu128 torch torchvision torchaudio

# Then install training stack
!pip install -q transformers datasets peft trl bitsandbytes accelerate sentencepiece

In [ ]:
# =========================
# IMPORTS + GPU CHECK
# =========================

import os
import re
import json
import math
import inspect
from pathlib import Path
from collections import Counter

import torch
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

print("Torch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("CUDA GPU not detected. QLoRA training for Qwen2.5-3B needs an NVIDIA GPU / Colab GPU.")

In [ ]:
# =========================
# CONFIG
# =========================

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# Upload this file to the notebook working directory, or update the path.
DATASET_PATH = "property_premise_dataset.jsonl"

# This normalized copy will be created by the validation cell.
NORMALIZED_DATASET_PATH = "property_premise_dataset_train_ready.jsonl"

OUTPUT_DIR = "./qwen-property-premise-generator-v1-r32"

# Premise generation outputs can be longer than opposing-counsel replies.
MAX_SEQ_LENGTH = 512

BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3

USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

PREMISE_SYSTEM_PROMPT = (
    "You generate realistic Indian property-law factual premises for legal training. "
    "Given a property-law topic, write one fact-rich dispute scenario with parties, "
    "timeline, documents, possession facts, evidence gaps, and litigation ambiguity. "
    "Do not provide legal analysis, advice, issues, conclusions, or judgments."
)

print("Config loaded")
print("Dataset path:", DATASET_PATH)
print("Output dir:", OUTPUT_DIR)
print("bf16:", USE_BF16)

In [ ]:
# =========================
# VALIDATE + NORMALIZE DATASET
# =========================
# Expected raw/final dataset format:
# {"messages":[{"role":"system",...},{"role":"user","content":"topic"},{"role":"assistant","content":"premise"}]}
#
# This cell validates the dataset and writes a normalized train-ready copy where:
# - every row is single-turn: system -> user -> assistant
# - system prompt is replaced by PREMISE_SYSTEM_PROMPT consistently
# - topic remains user content
# - premise remains assistant content

src = Path(DATASET_PATH)
assert src.exists(), f"Dataset not found: {DATASET_PATH}. Upload it or update DATASET_PATH."

def word_count(text: str) -> int:
    return len(re.findall(r"[\w'-]+", text))

def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text).strip().lower())

valid_rows = []
stats = Counter()
assistant_word_counts = []
user_values = []
seen_conversations = set()
seen_premises = set()

with src.open("r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        stats["total_lines"] += 1
        try:
            row = json.loads(line)
        except Exception:
            stats["invalid_json"] += 1
            continue

        if set(row.keys()) != {"messages"}:
            stats["bad_top_keys"] += 1
            continue

        messages = row.get("messages")
        if not isinstance(messages, list) or len(messages) != 3:
            stats["not_single_turn"] += 1
            continue

        roles = [m.get("role") for m in messages]
        if roles != ["system", "user", "assistant"]:
            stats["bad_role_order"] += 1
            continue

        if any(set(m.keys()) != {"role", "content"} for m in messages):
            stats["bad_message_keys"] += 1
            continue

        user_topic = str(messages[1].get("content", "")).strip()
        premise = str(messages[2].get("content", "")).strip()

        if not user_topic or not premise:
            stats["empty_content"] += 1
            continue

        wc = word_count(premise)
        if wc < 50:
            stats["assistant_under_50_words"] += 1
            continue
        if wc > 280:
            stats["assistant_over_280_words"] += 1
            continue

        # Premise generator should not output legal issue framing or analysis.
        low = premise.lower()
        if low.startswith("whether ") or "the issue is whether" in low:
            stats["issue_style_premise"] += 1
            continue
        if "therefore" in low and "court" in low and "hold" in low:
            stats["judgment_like"] += 1
            continue

        normalized_row = {
            "messages": [
                {"role": "system", "content": PREMISE_SYSTEM_PROMPT},
                {"role": "user", "content": user_topic},
                {"role": "assistant", "content": premise},
            ]
        }

        conv_key = normalize_text(json.dumps(normalized_row, ensure_ascii=False, sort_keys=True))
        premise_key = normalize_text(premise)
        if conv_key in seen_conversations or premise_key in seen_premises:
            stats["duplicates"] += 1
            continue

        seen_conversations.add(conv_key)
        seen_premises.add(premise_key)
        assistant_word_counts.append(wc)
        user_values.append(user_topic)
        valid_rows.append(normalized_row)

out = Path(NORMALIZED_DATASET_PATH)
with out.open("w", encoding="utf-8") as f:
    for row in valid_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "
")

print("Validation complete")
print("Input lines:", stats["total_lines"])
print("Valid train-ready rows:", len(valid_rows))
print("Rejected/flagged:", dict(stats))
print("Unique user topics:", len(set(user_values)))

if assistant_word_counts:
    print("Premise word count avg/min/max:", round(sum(assistant_word_counts)/len(assistant_word_counts), 2), min(assistant_word_counts), max(assistant_word_counts))

print("Top user topics:")
for topic, count in Counter(user_values).most_common(20):
    print(f"  {topic}: {count}")

print("
Normalized dataset written to:", NORMALIZED_DATASET_PATH)

In [ ]:
# =========================
# LOAD DATASET + TRAIN/VALIDATION SPLIT
# =========================

dataset = load_dataset(
    "json",
    data_files=NORMALIZED_DATASET_PATH,
    split="train",
)

print(dataset)
print(dataset[0])

dataset = dataset.train_test_split(test_size=0.05, seed=42)
print(dataset)

In [ ]:
# =========================
# LOAD TOKENIZER
# =========================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Tokenizer loaded")
print("EOS token:", tokenizer.eos_token)
print("PAD token:", tokenizer.pad_token)

# Quick chat template sanity check
sample_messages = dataset["train"][0]["messages"]
print(tokenizer.apply_chat_template(sample_messages, tokenize=False)[:1200])

In [ ]:
# =========================
# TOKEN LENGTH CHECK
# =========================
# Helps confirm MAX_SEQ_LENGTH is sufficient.

lengths = []
for ex in dataset["train"].select(range(min(500, len(dataset["train"])))):
    text = tokenizer.apply_chat_template(ex["messages"], tokenize=False)
    lengths.append(len(tokenizer(text, add_special_tokens=False)["input_ids"]))

lengths_sorted = sorted(lengths)

def pct(p):
    idx = int(len(lengths_sorted) * p / 100)
    idx = min(idx, len(lengths_sorted) - 1)
    return lengths_sorted[idx]

print("Token lengths on sample:")
print("min:", min(lengths_sorted))
print("avg:", round(sum(lengths_sorted) / len(lengths_sorted), 2))
print("p90:", pct(90))
print("p95:", pct(95))
print("p99:", pct(99))
print("max:", max(lengths_sorted))
print("MAX_SEQ_LENGTH:", MAX_SEQ_LENGTH)

if pct(99) > MAX_SEQ_LENGTH:
    print("WARNING: p99 exceeds MAX_SEQ_LENGTH. Increase MAX_SEQ_LENGTH to 1024 or expect truncation.")
else:
    print("MAX_SEQ_LENGTH looks safe for most samples.")

In [ ]:
# =========================
# LOAD BASE MODEL IN 4-BIT
# =========================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False
print("Model loaded in 4-bit")

In [ ]:
# =========================
# APPLY LoRA
# =========================

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# =========================
# SFT CONFIG
# =========================
# Uses messages column directly. TRL applies the model chat template.
# assistant_only_loss=True trains only on the premise output, not the prompt/topic.

sft_kwargs = dict(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    logging_steps=5,
    eval_steps=25,
    save_steps=50,
    save_total_limit=2,
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=USE_BF16,
    fp16=not USE_BF16,
    gradient_checkpointing=True,
    packing=True,
    assistant_only_loss=True,
    eos_token="<|im_end|>",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    report_to="none",
)

# Compatibility across TRL versions
sft_signature = inspect.signature(SFTConfig.__init__).parameters

if "max_seq_length" in sft_signature:
    sft_kwargs["max_seq_length"] = MAX_SEQ_LENGTH
elif "max_length" in sft_signature:
    sft_kwargs["max_length"] = MAX_SEQ_LENGTH

if "eval_strategy" in sft_signature:
    sft_kwargs["eval_strategy"] = "steps"
else:
    sft_kwargs["evaluation_strategy"] = "steps"

if "gradient_checkpointing_kwargs" in sft_signature:
    sft_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}

sft_config = SFTConfig(**sft_kwargs)
print("SFT config ready")
print(sft_config)

In [ ]:
# =========================
# TRAINER
# =========================

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    args=sft_config,
)

print("Trainer ready")

In [ ]:
# =========================
# TRAIN / AUTO-RESUME
# =========================

from transformers.trainer_utils import get_last_checkpoint

checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    checkpoint = get_last_checkpoint(OUTPUT_DIR)

if checkpoint:
    print(f"Resuming from checkpoint: {checkpoint}")
    trainer.train(resume_from_checkpoint=checkpoint)
else:
    print("Starting fresh training")
    trainer.train()

In [ ]:
# =========================
# SAVE LoRA ADAPTER + TOKENIZER
# =========================

trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved adapter/tokenizer to:", OUTPUT_DIR)

In [ ]:
# =========================
# PREMISE GENERATION TEST
# =========================
# Test the trained adapter directly in this runtime.

model.eval()

def generate_premise(topic: str, max_new_tokens: int = 260):
    messages = [
        {"role": "system", "content": PREMISE_SYSTEM_PROMPT},
        {"role": "user", "content": topic},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.85,
            top_p=0.9,
            repetition_penalty=1.12,
            eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>"),
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return generated.strip()

for topic in [
    "title dispute",
    "adverse possession",
    "RERA complaint",
    "forged sale deed",
    "tenant eviction",
]:
    print("="*90)
    print("TOPIC:", topic)
    print(generate_premise(topic))
    print()

In [ ]:
# =========================
# OPTIONAL: ZIP OUTPUT FOR DOWNLOAD
# =========================
# Useful in Colab.

import shutil

zip_path = shutil.make_archive(OUTPUT_DIR.replace("./", ""), "zip", OUTPUT_DIR)
print("Zipped model adapter:", zip_path)